# Yelp Veri Hazırlama ve Temizleme
Bu notebook, ham JSON verilerinden filtrelenmiş ve dengelenmiş bir CSV oluşturur.

In [ ]:
import os
import json
import pandas as pd
from sklearn.utils import resample
from tqdm.notebook import tqdm
import config

print("=== Başlangıç: Klasörleri Oluşturma ===")
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("features", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Dosya yolları
business_file = os.path.join("data", "yelp_academic_dataset_business.json")
review_file = os.path.join("data", "yelp_academic_dataset_review.json")
output_file = os.path.join("data", "reviews_cleaned.csv")


In [ ]:
print("\n=== ADIM 1: İşletme Verilerini Okuma ve Restoranları Filtreleme ===")
restaurant_ids = set()
for chunk in tqdm(pd.read_json(business_file, lines=True, chunksize=10000), desc="Business JSON"):
    mask = chunk['categories'].fillna('').str.contains('Restaurants', case=False, na=False)
    restaurants = chunk[mask]
    restaurant_ids.update(restaurants['business_id'].tolist())
    
print(f"Toplam bulunan restoran sayısı: {len(restaurant_ids)}")


In [ ]:
print("\n=== ADIM 2: Yorumları Okuma ve Sadece Restoranları Filtreleme ===")
filtered_reviews = []
columns_to_keep = ['review_id', 'business_id', 'stars', 'text', 'useful', 'funny', 'cool', 'date']

for chunk in tqdm(pd.read_json(review_file, lines=True, chunksize=10000), desc="Review JSON"):
    chunk_filtered = chunk[chunk['business_id'].isin(restaurant_ids)]
    chunk_filtered = chunk_filtered[columns_to_keep]
    filtered_reviews.append(chunk_filtered)
    
df = pd.concat(filtered_reviews, ignore_index=True)
toplam_ilk_yorum = len(df)
print(f"Filtrelenen toplam restoran yorumu sayısı: {toplam_ilk_yorum}")


In [ ]:
print("\n=== ADIM 3: Veri Temizliği ===")
# Boş veya None text at
df = df.dropna(subset=['text'])
df = df[df['text'].astype(str).str.strip() != '']

# 10 karakterden kısa yorumları at
df = df[df['text'].str.len() >= 10]

# Duplicate review_id'leri at
df = df.drop_duplicates(subset=['review_id'])

print(f"Temizlik sonrası kalan yorum sayısı: {len(df)}")


In [ ]:
print("\n=== ADIM 4: Yıldız -> Sınıf Dönüşümü ===")
def map_stars(star):
    if star in [1, 2]: return 0  # Kötü
    elif star == 3: return 1  # Orta
    elif star in [4, 5]: return 2  # İyi
    return -1

df['label'] = df['stars'].apply(map_stars)
df = df[df['label'] != -1]
print("Dönüşüm tamamlandı.")


In [ ]:
print("\n=== ADIM 5: Sınıf Dengesizliğini Kontrol Etme ve Undersampling ===")
print("Dengeleme Öncesi Sınıf Dağılımı:")
class_counts = df['label'].value_counts()
for label_val, count in class_counts.items():
    print(f"  {config.CLASS_NAMES[label_val]} (Sınıf {label_val}): {count} (%{count/len(df)*100:.2f})")
    
min_class_count = class_counts.min()
print(f"\nEn az sınıf sayısı ({min_class_count}) baz alınarak undersampling yapılıyor...")

dfs_to_concat = []
for label_val in class_counts.index:
    class_df = df[df['label'] == label_val]
    resampled_df = resample(class_df, replace=False, n_samples=min_class_count, random_state=config.RANDOM_STATE)
    dfs_to_concat.append(resampled_df)
    
df_balanced = pd.concat(dfs_to_concat)
df_balanced = df_balanced.sample(frac=1, random_state=config.RANDOM_STATE).reset_index(drop=True)

print("\nDengeleme Sonrası Sınıf Dağılımı:")
class_counts_balanced = df_balanced['label'].value_counts()
for label_val, count in class_counts_balanced.items():
    print(f"  {config.CLASS_NAMES[label_val]} (Sınıf {label_val}): {count} (%{count/len(df_balanced)*100:.2f})")


In [ ]:
print("\n=== ADIM 6: Veriyi CSV Olarak Kaydetme ===")
df_balanced.to_csv(output_file, index=False)
print(f"Dengelenmiş veri seti başarıyla kaydedildi: {output_file}")


In [ ]:
print("\n=== ADIM 7: Konsola İstatistik Raporu Yazdırma ===")
df_balanced['date'] = pd.to_datetime(df_balanced['date'])
df_balanced['text_len'] = df_balanced['text'].str.len()

print("\n" + "=" * 50)
print("VERİ SETİ İSTATİSTİK RAPORU".center(50))
print("=" * 50)
print(f"Toplam yorum sayısı (Temizlik Öncesi Filtrelenen): {toplam_ilk_yorum}")
print(f"Toplam yorum sayısı (Dengelenmiş Son Hali)       : {len(df_balanced)}")
print(f"\nBenzersiz İşletme Sayısı: {df_balanced['business_id'].nunique()} restoran")
print("=" * 50)
